[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_train_MA_A.ipynb)

# Part A: the final runs

Use a GPU runtime (**Runtime → Change runtime type → L4 GPU**). The cells clone the repository, install it, mount Drive, start TensorBoard and then run `train.py`, one call per run. The hyperparameters of every run are in `config.py`:

1. **Pretrain**: the final DAE (salt and pepper 0.5, h 256, dropout 0.4, 20 epochs, the epoch with the best val k-NN is kept) and the AE (the same, `--noise_level 0`).
2. **Frozen evaluation** of the DAE, the AE and a random encoder: k-NN (k=20, cosine), linear probe and analysis (A4 robustness, A5 evidence), on Tiny-ImageNet and CIFAR-100.
3. **Fine-tuning** of the same three encoders on 1%, 10% and 100% of the labels, on both datasets.
4. **Supervised baseline**: the same encoder trained from scratch with labels on 1%, 10% and 100%, and the frozen evaluation of the 100% one.

47 runs, seed 42, about 6 to 8 hours on an L4. Every run writes to `midterm_final` on Drive and is scored on the test set (only val chooses the checkpoint). After a disconnect, run all again: a finished run (it has `results.json`) is skipped, an unfinished one continues from its `last.ckpt`.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU. Use an L4 runtime.

In [1]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

Already up to date.
e5041ec (HEAD -> main, origin/main, origin/HEAD) Final values in config.py (DAE: dropout 0.4, 20 epochs; the tuned recipe for finetune and the supervised encoder); colab_train_MA_A.ipynb: plain train.py lines; remove colab_supervised_MA_A.ipynb; run_info.txt: the training time of the fit only; shorter comments


In [2]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

NVIDIA L4 | 12 CPU cores


## Drive
`OUT` is where every run writes.

In [3]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/RepL/Midterm"
OUT = f"{DRIVE_DIR}/midterm_final"
os.makedirs(OUT, exist_ok=True)
print("writes to:", OUT)

Mounted at /content/drive
writes to: /content/drive/MyDrive/RepL/Midterm/midterm_final


## TensorBoard
Filter the runs with part of a name (`pretrain`, `linear_probe`, `finetune`, `seed1`), then *Toggle all runs*. The `epoch` section has train, val and test on the same axis; the TEXT tab has each run's `run_info`.

In [4]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

<IPython.core.display.Javascript object>

## Settings
`BACKBONE` chooses the encoder of every run. The report uses the small encoder of the course (CNN-3); ResNet-18 is an option.

In [5]:
BACKBONE = "small"       # CNN-3, the course encoder: the runs of the report
# BACKBONE = "resnet18"  # optional: the same runs on ResNet-18 (config.py was tuned for CNN-3)
SEED = 42                # the training seed of every run
COMMON = f'--backbone {BACKBONE} --seed {SEED} --output_dir "{OUT}"'

ENCODERS = ("dae", "dae --noise_level 0", "random")  # DAE, AE, random
DATASETS = ("tinyimagenet", "cifar100")
FRACTIONS = (0.01, 0.1, 1.0)

## 1. Pretrain: the DAE and the AE

In [6]:
!python train.py --mode pretrain {COMMON}
!python train.py --mode pretrain --noise_level 0 {COMMON}

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..
DenoisingAutoEncoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=16384, out_features=256, b

## 2. Frozen evaluation: k-NN, linear probe, analysis
The linear probe uses the defaults of `config.py`.

In [7]:
for encoder in ENCODERS:
    for dataset in DATASETS:
        !python train.py --mode knn --encoder {encoder} --dataset {dataset} {COMMON}
        !python train.py --mode linear_probe --encoder {encoder} --dataset {dataset} {COMMON}
        !python train.py --mode analysis --encoder {encoder} --dataset {dataset} {COMMON}

Seed set to 42
I0000 00:00:1795602547.438376   14046 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795602547.511382   14046 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795602550.198417   14046 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795602550.199042   14046 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU wi

## 3. Fine-tuning on 1%, 10% and 100% of the labels
With the recipe tuned on val (`config.TUNED`); the learning rate of the new head is 3 times that of the encoder.

In [8]:
for encoder in ENCODERS:
    for dataset in DATASETS:
        for fraction in FRACTIONS:
            !python train.py --mode finetune --encoder {encoder} --dataset {dataset} --label_fraction {fraction} {COMMON}

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..
EncoderClassifier(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=16384, out_features=256, bias

## 4. The supervised baseline
The same encoder trained from scratch with labels (the same recipe), then the frozen evaluation of the one trained on 100%.

In [9]:
for fraction in FRACTIONS:
    !python train.py --mode pretrain_supervise_encoder --label_fraction {fraction} {COMMON}
for dataset in DATASETS:
    !python train.py --mode knn --encoder supervised --dataset {dataset} {COMMON}
    !python train.py --mode linear_probe --encoder supervised --dataset {dataset} {COMMON}
    !python train.py --mode analysis --encoder supervised --dataset {dataset} {COMMON}

Seed set to 42
date25112026_time041509_CNN3_final_supervised_1pct_lr0.003_wd1e-05_do0.5_aug-crop_jitter_ls0_opt-sgd_seed42 already finished; its results are in /content/drive/MyDrive/RepL/Midterm/midterm_final/date25112026_time041509_CNN3_final_supervised_1pct_lr0.003_wd1e-05_do0.5_aug-crop_jitter_ls0_opt-sgd_seed42/results.json

Test results:
  pretrain_supervise_encoder   (TinyImageNet): accuracy 0.039, top-5 0.122
Seed set to 42
date25112026_time043104_CNN3_final_supervised_10pct_lr0.003_wd1e-05_do0.5_aug-crop_jitter_ls0_opt-sgd_seed42 already finished; its results are in /content/drive/MyDrive/RepL/Midterm/midterm_final/date25112026_time043104_CNN3_final_supervised_10pct_lr0.003_wd1e-05_do0.5_aug-crop_jitter_ls0_opt-sgd_seed42/results.json

Test results:
  pretrain_supervise_encoder   (TinyImageNet): accuracy 0.142, top-5 0.326
Seed set to 42
date25112026_time044651_CNN3_final_supervised_100pct_lr0.003_wd1e-05_do0.5_aug-crop_jitter_ls0_opt-sgd_seed42 already finished; its results a